In [1]:
# Install and import necessary libraries
import heapq
!pip install datasets torch nltk rouge_score contractions
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader, Dataset

# Load the CNN/DailyMail dataset (version 3.0.0)
dataset = load_dataset("cnn_dailymail", "3.0.0")
# Reduce training set for prototyping
train_dataset = dataset["train"]
val_dataset   = dataset["validation"].shuffle(seed=42).select(range(5000))
test_dataset  = dataset["test"].shuffle(seed=42).select(range(100))

print(f"Train examples: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.9/113.9 kB 9.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b208cbaf6decfe37739f85ccca70dfd69fc1e9c110e38d04f0e1f6ebbc68bc6

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Train examples: 287113, Validation: 5000, Test: 100


In [2]:
import re
import nltk
import contractions
import json
from collections import Counter

nltk.download('punkt')

# ---------------------------------------
# TEXT CLEANING FUNCTION
# ---------------------------------------

def clean_text(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  # Keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()  # Normalize whitespace
    return text

print(clean_text("Here's an example: Don't remove 100% of letters."))

# ---------------------------------------
# BUILD VOCABULARY FROM DATASET
# ---------------------------------------

min_freq = 5           # Minimum frequency to keep a word
vocab_size_limit = 50000  # Max vocabulary size

counter = Counter()
for item in train_dataset:
    article_tokens = nltk.word_tokenize(clean_text(item["article"]))
    summary_tokens = nltk.word_tokenize(clean_text(item["highlights"]))
    counter.update(article_tokens)
    counter.update(summary_tokens)

# Apply frequency threshold, sort, and cap by vocab size
filtered_words = [word for word, freq in counter.items() if freq >= min_freq]
most_common = filtered_words[:vocab_size_limit]

# Special tokens
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
START_TOKEN = "<start>"
END_TOKEN = "<end>"
special_tokens = [PAD_TOKEN, UNK_TOKEN, START_TOKEN, END_TOKEN]

# Build word2idx and idx2word dictionaries
word2idx = {tok: idx for idx, tok in enumerate(special_tokens)}
for word in most_common:
    if word not in word2idx:
        word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)
print(f"✅ Vocabulary size (with special tokens): {vocab_size}")

# Save vocab mappings for reproducibility
with open("word2idx.json", "w") as f:
    json.dump(word2idx, f)
with open("idx2word.json", "w") as f:
    json.dump(idx2word, f)

# ---------------------------------------
# SEQUENCE PADDING FUNCTION
# ---------------------------------------

def pad_sequence(seq, pad_value, max_len):
    return seq[:max_len] + [pad_value] * max(0, max_len - len(seq))

# ---------------------------------------
# ENCODING FUNCTIONS
# ---------------------------------------

def encode_article(text, max_len=600):
    tokens = nltk.word_tokenize(clean_text(text))
    idxs = [word2idx.get(tok, word2idx[UNK_TOKEN]) for tok in tokens]
    return pad_sequence(idxs, word2idx[PAD_TOKEN], max_len)

def encode_summary(text, max_len=150):
    tokens = nltk.word_tokenize(clean_text(text))
    tokens = tokens[:(max_len - 2)]  # Reserve space for <start> and <end>

    input_ids = [word2idx[START_TOKEN]] + [word2idx.get(tok, word2idx[UNK_TOKEN]) for tok in tokens]
    target_ids = [word2idx.get(tok, word2idx[UNK_TOKEN]) for tok in tokens] + [word2idx[END_TOKEN]]

    input_ids = pad_sequence(input_ids, word2idx[PAD_TOKEN], max_len)
    target_ids = pad_sequence(target_ids, word2idx[PAD_TOKEN], max_len)

    return input_ids, target_ids

# ---------------------------------------
# TEST ENCODING ON A SAMPLE
# ---------------------------------------

sample_article = train_dataset[0]["article"]
sample_summary = train_dataset[0]["highlights"]

art_ids = encode_article(sample_article)
summ_in, summ_out = encode_summary(sample_summary)

# Assert expected lengths
assert len(art_ids) == 600
assert len(summ_in) == 150
assert len(summ_out) == 150

print("✅ Encoding test passed. Lengths are correct.")


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


here is an example do not remove of letters
✅ Vocabulary size (with special tokens): 50004
✅ Encoding test passed. Lengths are correct.


In [3]:
import os
import urllib.request
import zipfile
import torch
import ssl
# ------------------------------
# GloVe Download
# ------------------------------
glove_url = "http://nlp.stanford.edu/data/glove.6B.zip"
glove_zip = "glove.6B.zip"
glove_file = "glove.6B.300d.txt"
ssl_context = ssl._create_unverified_context()

if not os.path.exists(glove_file):
    print("🔽 Downloading GloVe embeddings...")
    with urllib.request.urlopen(glove_url, context=ssl_context) as response:
        with open(glove_zip, 'wb') as out_file:
            out_file.write(response.read())
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extract(glove_file, '.')
    print("✅ GloVe downloaded and extracted.")
else:
    print("📁 GloVe already exists. Skipping download.")

# ------------------------------
# Load GloVe Vectors
# ------------------------------
print("📦 Loading GloVe vectors...")
glove_embeddings = {}
with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.strip().split()
        word = values[0]
        vector = torch.tensor(list(map(float, values[1:])), dtype=torch.float)
        glove_embeddings[word] = vector
print(f"✅ Loaded {len(glove_embeddings):,} GloVe vectors.")

# ------------------------------
# Build Embedding Matrix
# ------------------------------
embedding_dim = 300
embedding_matrix = torch.zeros((vocab_size, embedding_dim))

for word, idx in word2idx.items():
    if word == PAD_TOKEN:
        continue
    elif word in [UNK_TOKEN, START_TOKEN, END_TOKEN]:
        torch.nn.init.xavier_uniform_(embedding_matrix[idx].unsqueeze(0))
    elif word.lower() in glove_embeddings:
        embedding_matrix[idx] = glove_embeddings[word.lower()]
    else:
        torch.nn.init.xavier_uniform_(embedding_matrix[idx].unsqueeze(0))

# ------------------------------
# Create Frozen Embedding Layer
# ------------------------------
embedding_layer = torch.nn.Embedding.from_pretrained(
    embedding_matrix,
    freeze=True,
    padding_idx=word2idx[PAD_TOKEN]
)

print(f"✅ Embedding matrix shape: {embedding_matrix.shape}")

# ------------------------------
# Sample Encoding
# ------------------------------
art_ids = encode_article(train_dataset[0]["article"])
summ_in, summ_out = encode_summary(train_dataset[0]["highlights"])
print(f"🔢 Sample lengths – article: {len(art_ids)}, summary input: {len(summ_in)}, summary output: {len(summ_out)}")




🔽 Downloading GloVe embeddings...
✅ GloVe downloaded and extracted.
📦 Loading GloVe vectors...
✅ Loaded 400,000 GloVe vectors.
✅ Embedding matrix shape: torch.Size([50004, 300])
🔢 Sample lengths – article: 600, summary input: 150, summary output: 150


In [4]:
train_subset = dataset["train"].shuffle(seed=42).select(range(50000))
class MyCustomDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # Return whatever fields your model expects
        # Example: article and highlights
        item = self.dataset[idx]
        article = item["article"]
        summary = item["highlights"]
        return {"article": article, "highlights": summary}
 # Or encode if needed

# Create dataset object
train_data_1 = MyCustomDataset(train_subset)


In [5]:
from torch.utils.data import Dataset, DataLoader

class SummarizationDataset(Dataset):
    def __init__(self, data_split):
        self.data = data_split  # raw HF dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        article = encode_article(self.data[idx]["article"])
        summary_in, summary_out = encode_summary(self.data[idx]["highlights"])
        return {
            "article": torch.tensor(article, dtype=torch.long),
            "summary_in": torch.tensor(summary_in, dtype=torch.long),
            "summary_out": torch.tensor(summary_out, dtype=torch.long)
        }

# Create datasets
train_dataset_prep = SummarizationDataset(train_data_1)
val_dataset_prep   = SummarizationDataset(val_dataset)
test_dataset_prep  = SummarizationDataset(test_dataset)

# Create dataloaders
train_loader = DataLoader(train_dataset_prep, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset_prep, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset_prep, batch_size=16, shuffle=False)

print("DataLoaders ready:")
print(f"- Train: {len(train_loader)} batches")
print(f"- Val:   {len(val_loader)} batches")
print(f"- Test:  {len(test_loader)} batches")


DataLoaders ready:
- Train: 3125 batches
- Val:   313 batches
- Test:  7 batches


In [6]:
import torch.nn as nn
import torch.nn.functional as F

# Luong attention mechanism
class LuongAttention(nn.Module):
    def __init__(self, method, hidden_size):
        super(LuongAttention, self).__init__()
        assert method in ['dot', 'general', 'concat']
        self.method = method
        self.hidden_size = hidden_size
        if method == 'general':
            self.attn = nn.Linear(hidden_size, hidden_size)
        elif method == 'concat':
            self.attn = nn.Linear(hidden_size * 2, hidden_size)
            self.v = nn.Parameter(torch.FloatTensor(hidden_size))

    def dot_score(self, hidden, encoder_output):
        # hidden: (1, batch, hidden_dim)
        # encoder_output: (batch, seq_len, hidden_dim)
        hidden = hidden.squeeze(0)  # (batch, hidden_dim)
        return torch.sum(encoder_output * hidden.unsqueeze(1), dim=2)  # (batch, seq_len)

    def general_score(self, hidden, encoder_output):
        energy = self.attn(encoder_output)
        hidden = hidden.squeeze(0)
        return torch.sum(hidden.unsqueeze(1) * energy, dim=2)

    def concat_score(self, hidden, encoder_output):
        expanded_hidden = hidden.expand(encoder_output.size(0), -1, -1)  # (batch, seq_len, hidden)
        energy = torch.tanh(self.attn(torch.cat((expanded_hidden, encoder_output), 2)))  # (batch, seq_len, hidden)
        return torch.sum(self.v * energy, dim=2)  # (batch, seq_len)

    def forward(self, hidden, encoder_outputs):
        if self.method == 'general':
            attn_energies = self.general_score(hidden, encoder_outputs)
        elif self.method == 'concat':
            attn_energies = self.concat_score(hidden, encoder_outputs)
        else:  # dot
            attn_energies = self.dot_score(hidden, encoder_outputs)
        attn_weights = F.softmax(attn_energies, dim=1).unsqueeze(1)  # (batch, 1, seq_len)
        return attn_weights

import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, embedding, hidden_size, n_layers=2, dropout=0.2):
        super(Encoder, self).__init__()
        self.embedding = embedding
        self.hidden_size = hidden_size
        self.n_layers = n_layers

        self.lstm = nn.LSTM(input_size=embedding.embedding_dim,
                            hidden_size=hidden_size,
                            num_layers=n_layers,
                            dropout=(0 if n_layers == 1 else dropout),
                            bidirectional=True,
                            batch_first=True)

    def forward(self, input_seq, input_lengths):
        embedded = self.embedding(input_seq)  # (batch, seq, embed_dim)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, input_lengths,
                                                   batch_first=True, enforce_sorted=False)
        outputs, (h, c) = self.lstm(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)

        # Concatenate fw and bw
        H = self.hidden_size
        outputs = torch.cat((outputs[:, :, :H], outputs[:, :, H:]), dim=2)  # (batch, seq, 2*H)

        # Reshape hidden/cell: (n_layers, 2, B, H) → concat → (n_layers, B, 2H)
        h = h.view(self.n_layers, 2, -1, H)
        c = c.view(self.n_layers, 2, -1, H)
        h = torch.cat((h[:, 0], h[:, 1]), dim=2)
        c = torch.cat((c[:, 0], c[:, 1]), dim=2)

        return outputs, (h, c)



class Decoder(nn.Module):
    def __init__(self, embedding, hidden_size, output_size, attn_method='dot', n_layers=2, dropout=0.2):
        super(Decoder, self).__init__()
        self.embedding = embedding
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers

        self.embedding_dropout = nn.Dropout(dropout)

        # input_size = embedding_dim
        self.lstm = nn.LSTM(input_size=embedding.embedding_dim,
                            hidden_size=hidden_size * 2,  # match encoder hidden output (2*H)
                            num_layers=n_layers,
                            dropout=(0 if n_layers == 1 else dropout),
                            batch_first=False)

        self.attn = LuongAttention(attn_method, hidden_size * 2)
        self.concat = nn.Linear((hidden_size * 2) * 2, hidden_size * 2)
        self.out = nn.Linear(hidden_size * 2, output_size)

    def forward(self, input_step, last_hidden, encoder_outputs):
        embedded = self.embedding(input_step)  # (1, B, E)
        embedded = self.embedding_dropout(embedded)

        rnn_output, hidden = self.lstm(embedded, last_hidden)  # rnn_output: (1, B, 2H)

        attn_weights = self.attn(rnn_output, encoder_outputs)  # (B, 1, T)
        context = attn_weights.bmm(encoder_outputs)            # (B, 1, 2H)
        context = context.permute(1, 0, 2)                     # (1, B, 2H)

        rnn_output = rnn_output.squeeze(0)  # (B, 2H)
        context = context.squeeze(0)        # (B, 2H)

        concat_input = torch.cat((rnn_output, context), dim=1)  # (B, 4H)
        concat_output = torch.tanh(self.concat(concat_input))   # (B, 2H)

        output = self.out(concat_output)                        # (B, vocab_size)
        output = F.log_softmax(output, dim=1)
        return output, hidden

# Instantiate encoder and decoder
hidden_size = 256
encoder = Encoder(embedding_layer, hidden_size)
decoder = Decoder(embedding_layer, hidden_size, output_size=vocab_size, attn_method='dot')



In [7]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import random

pad_idx = word2idx[PAD_TOKEN]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder.to(device)
decoder.to(device)
embedding_layer.weight.requires_grad = False

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
criterion = nn.NLLLoss(ignore_index=pad_idx, reduction='sum')  # summed loss

num_epochs = 3
teacher_forcing_ratio = 0.5
best_val_loss = float('inf')

for epoch in range(1, num_epochs + 1):
    encoder.train()
    decoder.train()
    train_loss = 0.0
    train_tokens = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        articles = batch["article"].to(device)
        summ_in = batch["summary_in"].to(device)
        summ_out = batch["summary_out"].to(device)

        lengths = (articles != pad_idx).sum(dim=1).cpu()
        encoder_outputs, (h, c) = encoder(articles, lengths)

        decoder_input = summ_in[:, 0].unsqueeze(0)  # (1, batch)
        decoder_hidden = (h, c)
        batch_loss = 0.0
        batch_tokens = 0

        optimizer.zero_grad()

        for t in range(1, summ_in.size(1)):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_outputs)
            target = summ_out[:, t]

            non_pad_mask = (target != pad_idx)
            if non_pad_mask.any():
                loss = criterion(decoder_output, target)
                batch_loss += loss
                batch_tokens += non_pad_mask.sum().item()

            # Teacher forcing
            use_tf = random.random() < teacher_forcing_ratio
            if use_tf:
                decoder_input = summ_in[:, t].unsqueeze(0)
            else:
                top1 = decoder_output.argmax(1)
                decoder_input = top1.unsqueeze(0)

        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
        optimizer.step()

        train_loss += batch_loss.item()
        train_tokens += batch_tokens

    scheduler.step()

    # Validation
    encoder.eval()
    decoder.eval()
    val_loss = 0.0
    val_tokens = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            articles = batch["article"].to(device)
            summ_in = batch["summary_in"].to(device)
            summ_out = batch["summary_out"].to(device)

            lengths = (articles != pad_idx).sum(dim=1).cpu()
            encoder_outputs, (h, c) = encoder(articles, lengths)

            decoder_input = summ_in[:, 0].unsqueeze(0)
            decoder_hidden = (h, c)

            for t in range(1, summ_in.size(1)):
                decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_outputs)
                target = summ_out[:, t]

                non_pad_mask = (target != pad_idx)
                if non_pad_mask.any():
                    loss = criterion(decoder_output, target)
                    val_loss += loss.item()
                    val_tokens += non_pad_mask.sum().item()

                decoder_input = summ_in[:, t].unsqueeze(0)

    avg_train_loss = train_loss / train_tokens
    avg_val_loss = val_loss / val_tokens
    print(f"\nEpoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'encoder': encoder.state_dict(),
            'decoder': decoder.state_dict(),
            'word2idx': word2idx,
            'idx2word': idx2word
        }, "best_model.pt")


/usr/local/lib/python3.11/dist-packages/torch/_compile.py:32: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


Epoch 1 [Train]:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 1 [Val]:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 1: Train Loss=7.3788, Val Loss=7.1326


Epoch 2 [Train]:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 2 [Val]:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 2: Train Loss=7.1156, Val Loss=6.9036


Epoch 3 [Train]:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 3 [Val]:   0%|          | 0/313 [00:00<?, ?it/s]


Epoch 3: Train Loss=6.9482, Val Loss=6.7803


In [8]:
import pickle
with open("vocab.pkl", "wb") as f:
    pickle.dump({'word2idx': word2idx, 'idx2word': idx2word}, f)


In [9]:
import torch
import torch.nn.functional as F

# Load best model
checkpoint = torch.load("best_model.pt", map_location=device)
encoder.load_state_dict(checkpoint['encoder'])
decoder.load_state_dict(checkpoint['decoder'])
encoder.eval()
decoder.eval()

def beam_search_decode(enc_outputs, decoder_hidden,
                       beam_width=5, max_len=150,
                       length_penalty=0.9, repetition_penalty=0.8):
    start_token = word2idx[START_TOKEN]
    end_token   = word2idx[END_TOKEN]
    sequences   = [([start_token], 0.0, decoder_hidden)]
    completed   = []

    for _ in range(max_len):
        next_sequences = []
        for seq, score, hidden in sequences:
            if seq[-1] == end_token:
                completed.append((seq, score / (len(seq) ** length_penalty)))
                continue

            decoder_input = torch.tensor([[seq[-1]]], device=device)
            dec_out, hidden_new = decoder(decoder_input, hidden, enc_outputs)
            topv, topi = dec_out.squeeze(0).topk(beam_width)

            for i in range(beam_width):
                token = topi[i].item()
                # apply repetition penalty
                rep_pen = repetition_penalty if token in seq else 0.0
                new_score = score + topv[i].item() - rep_pen
                new_seq   = seq + [token]

                if token == end_token:
                    norm_score = new_score / (len(new_seq) ** length_penalty)
                    completed.append((new_seq, norm_score))
                else:
                    # clone hidden states
                    h_new = hidden_new[0].clone()
                    c_new = hidden_new[1].clone()
                    next_sequences.append((new_seq, new_score, (h_new, c_new)))

        if not next_sequences:
            break

        # keep top-k sequences
        next_sequences.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        sequences = next_sequences[:beam_width]

    # finalize incomplete sequences
    for seq, score, _ in sequences:
        if seq[-1] != end_token:
            norm_score = score / (len(seq) ** length_penalty)
            completed.append((seq, norm_score))

    # choose best
    if not completed:
        return [end_token]
    completed.sort(key=lambda x: x[1], reverse=True)
    best_seq = completed[0][0]

    # strip start/end tokens
    end_pos = best_seq.index(end_token) if end_token in best_seq else len(best_seq)
    return best_seq[1:end_pos]

def summarize(article, beam_width=5):
    art_ids   = encode_article(article)
    art_tensor= torch.tensor(art_ids, dtype=torch.long).unsqueeze(0).to(device)
    length    = torch.tensor([len([w for w in art_ids if w != word2idx[PAD_TOKEN]])])

    with torch.no_grad():
        enc_outputs, (h, c) = encoder(art_tensor, length)

    # perform beam search with default rep_penalty=0.8
    best_sequence = beam_search_decode(enc_outputs, (h, c), beam_width=beam_width)

    # convert to words
    summary = ' '.join(idx2word[idx] for idx in best_sequence if idx != word2idx[PAD_TOKEN])
    return summary

# Example usage
test_article = test_dataset[0]["article"]
print("📰 Article (truncated):", test_article[:100], "…")
print("✅ Reference Summary:", test_dataset[0]["highlights"])
print("📝 Generated Summary:", summarize(test_article, beam_width=5))


📰 Article (truncated): (CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on  …
✅ Reference Summary: CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for too long"
📝 Generated Summary: <unk> is to the in of and to a of the says says says are to to to the to the the says says is to to the to the


In [10]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from tqdm.auto import tqdm

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

total_rouge1 = total_rouge2 = total_rougeL = 0.0
total_bleu = 0.0
count = 0

encoder.eval()
decoder.eval()

for batch in tqdm(test_loader, desc="Evaluating"):
    articles = batch["article"].numpy()
    summaries = batch["summary_out"].numpy()

    for i in range(len(articles)):
        # Decode article to text
        art_ids = [w for w in articles[i].tolist() if w != word2idx[PAD_TOKEN]]
        article_text = ' '.join(idx2word.get(idx, '<unk>') for idx in art_ids)

        # Generate summary using beam search
        gen_summary = summarize(article_text, beam_width=5)

        # Prepare reference summary
        ref_ids = summaries[i].tolist()
        if word2idx[END_TOKEN] in ref_ids:
            ref_ids = ref_ids[:ref_ids.index(word2idx[END_TOKEN])]
        reference = [idx2word.get(idx, '<unk>') for idx in ref_ids if idx != word2idx[PAD_TOKEN]]
        reference_text = ' '.join(reference)

        # Compute ROUGE scores
        scores = scorer.score(reference_text, gen_summary)
        total_rouge1 += scores['rouge1'].fmeasure
        total_rouge2 += scores['rouge2'].fmeasure
        total_rougeL += scores['rougeL'].fmeasure

        # Compute BLEU score
        gen_tokens = gen_summary.split()
        ref_tokens = reference

        if len(gen_tokens) == 0:
            bleu = 0.0
        else:
            bleu = sentence_bleu([ref_tokens], gen_tokens)

        total_bleu += bleu
        count += 1

# Final average scores
avg_rouge1 = total_rouge1 / count
avg_rouge2 = total_rouge2 / count
avg_rougeL = total_rougeL / count
avg_bleu   = total_bleu / count

print("\n📊 Evaluation Metrics:")
print(f"ROUGE-1: {avg_rouge1:.4f}")
print(f"ROUGE-2: {avg_rouge2:.4f}")
print(f"ROUGE-L: {avg_rougeL:.4f}")
print(f"BLEU    : {avg_bleu:.4f}")


Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_


📊 Evaluation Metrics:
ROUGE-1: 0.1983
ROUGE-2: 0.0156
ROUGE-L: 0.1434
BLEU    : 0.0000
